## Integer Unpacking benchmark results

The following interaction explores the results of the benchmarks on various architectures.
The controls let you set with the following options.

#### Functions:
The different implementations being benchmark.
This is the combination of the micro architecture and the specific code.
The micro architecture such are `Neon` on `Arm64` and `SSE4.2`, `AVX2` and `AVX512` on `x86-64`.
The different code functions are:
 - **ScalarExact**: The naive function that can unpack exactly the desired number of values and that is used to wrap other fuctions that can only extract a batch of values at the time. 
 - **ScalarBatch**: A version without SIMD code that extract values in batch.
 - **Old**: The SIMD version that was previously in Apache Arrow.
 - **New**: The new SIMD implementation introduce with the work presented here.
 - **NewNoDispatch**: The same SIMD implementation, with all working types fixed to the one described.
 - **LitteIntPacker**: The work by Daniel Lemire available in the [LittleIntPacker library](https://github.com/fast-pack/LittleIntPacker) (under horizontal packing). This was added to this build for benchmark and wraped with the same bit width dispatch mechanism for a fair comparison. This is only available for the `SSE` family on `x86-64` and for `uint32_t`, but we added a few more unpacking type by extracting in a local buffer and looping on it with `static_cast` in the same way we do used for casting between different integer types.
 - **Dynamic**: Should be the best function for the architecture, wrapped in a dynamic dispatch to the apporpriate function depending on the micro architecture detected at runtime.

*Note*: The scalar functions are compiled in different files with different for different micro architectures to evaluate the capacity of the compiler to auto-vectorize the code compared the manual vectoization we do with the [xsimd library](https://github.com/xtensor-stack/xsimd/). 
Although `Neon` is always available on `Arm64`, we specifically built some files without it to isolate what the compiler auto vectorization can do.
This setting is intended for research and does not make sense in general.

#### Unpacked type:
This is the type the that the unpack function targets.

#### Packed width:
This is the the number of bits used to encode individual values.
Larger bit width can only be extracted to larger integer types.

In [ ]:
import pathlib

import polars as pl
import ipywidgets as widgets

import results_dashboard.preprocessing as preprocessing
import results_dashboard.dashboard as dashboard


def load_cleaned_benchmarks(
    arch: str = "linux-64", directory: pathlib.Path | str = "data"
) -> pl.DataFrame:
    raw_path = pathlib.Path(directory) / f"benchmark-{arch}.csv"
    processed_path = pathlib.Path(directory) / f"benchmark-{arch}.parquet"

    if preprocessing.is_emscripten() or processed_path.exists():
        return preprocessing.read_parquet(processed_path)

    # Variable number of rows to skip at the top
    n_comment_lines = count_header_lines(raw_path)
    lf_raw = pl.scan_csv(raw_path, skip_rows=n_comment_lines)
    df = preprocessing.clean_benchmark(lf_raw).collect()
    df.write_parquet(processed_path)
    return df


def make_raw_results_explorer(df: pl.DataFrame):
    arch_funcs = dashboard.make_arch_funcs(df)
    palette = dashboard.build_palette(arch_funcs)
    dashes = dashboard.build_dashes(arch_funcs)

    unpacked_type_wt = dashboard.make_unpacked_type_wt(df=df)
    func_wt = dashboard.make_func_wt(arch_funcs)
    packed_width_one_wt, packed_width_one_slider_wt = (
        dashboard.make_packed_width_one_pair_wt(unpacked_type_wt, df=df)
    )
    x_axis_wt = dashboard.make_x_axis_wt(df=df)
    out = widgets.Output()

    def plot_callback(*args, **kwargs):
        dashboard.raw_plot(
            df=df,
            unpacked_types=unpacked_type_wt.value,
            packed_width=packed_width_one_wt.value,
            arch_funcs=func_wt.value,
            x_axis=x_axis_wt.value,
            out=out,
            palette=palette,
            dashes=dashes,
            aspect=1.5,
            facet_kws={"sharex": False},
        )

    x_axis_wt.observe(plot_callback, names="value")
    unpacked_type_wt.observe(plot_callback, names="value")
    packed_width_one_wt.observe(plot_callback, names="value")
    func_wt.observe(plot_callback, names="value")

    controls = widgets.Tab(
        children=[
            # Functions
            widgets.VBox(
                [
                    widgets.Label("Select all function to compare:"),
                    func_wt,
                ]
            ),
            # Unpacked width
            widgets.VBox(
                [
                    widgets.Label("Select the target type for unpacking:"),
                    unpacked_type_wt,
                ],
                layout=widgets.Layout(min_height="100px"),
            ),
            # Packed width
            widgets.VBox(
                [
                    widgets.Label("Packed Width (in bits):"),
                    widgets.HBox([packed_width_one_slider_wt, packed_width_one_wt]),
                ]
            ),
            x_axis_wt,
        ],
        titles=["Functions", "Unpacked Type", "Packed Width", "View"],
        layout=widgets.Layout(min_height="200px"),
    )

    return widgets.VBox([controls, out]), plot_callback


def make_speed_results_explorer(df: pl.DataFrame):
    arch_funcs = dashboard.make_arch_funcs(df)
    palette = dashboard.build_palette(arch_funcs)
    dashes = dashboard.build_dashes(arch_funcs)

    unpacked_type_wt = dashboard.make_unpacked_type_wt(df=df)
    func_wt = dashboard.make_func_wt(arch_funcs)
    out = widgets.Output()

    def plot_callback(*args, **kwargs):
        dashboard.plot_speed(
            df=df,
            unpacked_types=unpacked_type_wt.value,
            arch_funcs=func_wt.value,
            out=out,
            palette=palette,
            dashes=dashes,
            aspect=1.5,
            facet_kws={"sharex": False},
        )

    unpacked_type_wt.observe(plot_callback, names="value")
    func_wt.observe(plot_callback, names="value")

    controls = widgets.Tab(
        children=[
            # Functions
            widgets.VBox(
                [
                    widgets.Label("Select all function to compare:"),
                    func_wt,
                ]
            ),
            # Unpacked width
            widgets.VBox(
                [
                    widgets.Label("Select the target type for unpacking:"),
                    unpacked_type_wt,
                ],
                layout=widgets.Layout(min_height="100px"),
            ),
        ],
        titles=["Functions", "Unpacked Type"],
        layout=widgets.Layout(min_height="200px"),
    )

    return widgets.VBox([controls, out]), plot_callback

### Full results

In [ ]:
raw_arch_wt = widgets.ToggleButtons(options=["linux-64", "osx-arm64"])
raw_controls = widgets.VBox([])


def raw_arch_callback(*args, **kwargs):
    df = load_cleaned_benchmarks(raw_arch_wt.value)
    wt, start = make_raw_results_explorer(df)
    raw_controls.children = [raw_arch_wt, wt]
    start()


raw_arch_wt.observe(raw_arch_callback, names="value")
raw_arch_callback()

raw_controls

### Speed agregation

In [ ]:
speed_arch_wt = widgets.ToggleButtons(options=["linux-64", "osx-arm64"])
speed_controls = widgets.VBox([])


def speed_arch_callback(*args, **kwargs):
    df = load_cleaned_benchmarks(speed_arch_wt.value)
    wt, start = make_speed_results_explorer(df=dashboard.linear_regression(df))
    speed_controls.children = [speed_arch_wt, wt]
    start()


speed_arch_wt.observe(speed_arch_callback, names="value")
speed_arch_callback()

speed_controls